# load lib

In [1]:
suppressPackageStartupMessages({
    library(Seurat)
    library(ggplot2)
    library(dplyr)
    library(ggthemes)
    library(uwot)
    library(pheatmap)
    library(harmony)
    library(tibble)
    library(tidyr)
    library(rlang)
    library(purrr)
    library(glue)
    library(jsonlite)
    source('/data/srlab/AMP_collab/lakshay-yakir/_common/typing_utils.r')
})

# create sc allcells 

In [2]:
basepath = '/data/srlab/AMP_collab/AMP2_2023_seuratObj_galoz/'

In [16]:
process_AMPp2_obj <- function(ct) {
    sc <- readRDS(file.path(basepath, paste0('AMP2_seuratObj_', ct, '.202511.rds')))
    counts <- LayerData(sc, assay = 'RNA', layer = 'counts')
    data <- LayerData(sc, assay = 'RNA', layer = 'data')
    sc <- CreateSeuratObject(
        counts = counts, 
        meta.data = sc[[]]
        )
    sc[['RNA']]$data = data 
    
    sc$cohort = 'AMPp2'
    sc$lineage = sc[[]] %>% 
        mutate(lineage = case_when(
            grepl("^B-", cluster_name) ~ 'B_plasma', 
            grepl("^M-", cluster_name) ~ 'Myeloid', 
            grepl("(^T|^NK)-", cluster_name) ~ 'T_NK', 
            grepl("^E-", cluster_name) ~ 'Endothelial', 
            .default = 'Stromal'
        )
               ) %>% 
        pull(lineage)
    
    sc$celltypes.med <- sc[[]] %>% 
        mutate(celltypes.med = case_when(
        grepl('^B-[6-8]', cluster_name)                        ~ 'Plasma',
        grepl('^B-',      cluster_name)                        ~ 'B',
        grepl('^E-[0-3]', cluster_name)                        ~ 'Vascular endothelial',
        grepl('^E-',      cluster_name)                        ~ 'Lymphatic endothelial',
        grepl('sublining|intermediate', cluster_name)          ~ 'Sublining',
        grepl('lining',   cluster_name)                        ~ 'Lining',
        grepl('^M-[0-8]:', cluster_name)                       ~ 'Macrophage',
        grepl('^M-', cluster_name)                             ~ 'Dendritic cell', 
        grepl('^T-', cluster_name)                             ~ 'T',
        grepl('^NK-', cluster_name)                            ~ 'NK',
        .default                                               = 'Mural'
      ))
    return(sc)
}

In [17]:
endo <- process_AMPp2_obj('endothelial')
fib <- process_AMPp2_obj('fibroblast') 
bplasma <- process_AMPp2_obj('Bcell')
mye <- process_AMPp2_obj('myeloid') 
t <- process_AMPp2_obj('T')
nk <- process_AMPp2_obj('NK') 

In [18]:
sc_list = list(endo, fib, bplasma, mye, t, nk)
shared_cols = Reduce(intersect, lapply(sc_list, function(obj) colnames(obj@meta.data)))
sc_list = lapply(sc_list, function(sc) {
    sc@meta.data = sc@meta.data[, shared_cols]
    return(sc)
})
sc <- merge(sc_list[[1]], y = sc_list[-1])  
sc <- JoinLayers(sc)                                      
sc

An object of class Seurat 
33538 features across 314011 samples within 1 assay 
Active assay: RNA (33538 features, 0 variable features)
 2 layers present: data, counts

In [19]:
table(sc$lineage)
table(sc$celltypes.med)


   B_plasma Endothelial     Myeloid     Stromal        T_NK 
      30691       25043       76181       79555      102541 


                    B        Dendritic cell                Lining 
                21580                 15121                 25862 
Lymphatic endothelial            Macrophage                 Mural 
                  406                 61060                  1736 
                   NK                Plasma             Sublining 
                 8495                  9111                 51957 
                    T  Vascular endothelial 
                94046                 24637 

In [21]:
saveRDS(sc, '/data/srlab/AMP_collab/AMP2_2023_seuratObj_galoz/AMPp2_seuratObj_allcells.202511.rds')